# Лабораторная работа №1
## Компьютерная геометрия

**Цель работы:** знакомство с параметрическим представлением прямой, отрезков, кривых. Проверка знаний основ библиотек NumPy и Matplotlib.

**Студент:** ___________________________________

**Дата выполнения:** ___________________________

---

## Требования к оформлению (из 00.pdf и 01.pdf)

| Требование | Как проверяется |
|------------|-----------------|
| Объектно-ориентированный интерфейс Matplotlib (`ax.`, `fig.`) | В коде нет `plt.plot()`, только `ax.plot()` |
| Сетка координат | `ax.grid(True)` |
| Одинаковый масштаб осей | `ax.set_aspect('equal')` |
| Размер и разрешение | `figsize=(5,5)`, `dpi=200` |
| Интерактивные ползунки | `ipywidgets.interact` |
| Формулы в математическом виде | LaTeX в Markdown-ячейках |

---

In [ ]:
# Импорт всех необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ipywidgets import interact, FloatSlider, IntSlider

# Настройка интерактивных графиков в Jupyter
%matplotlib widget

# Библиотеки, которые могут понадобиться в следующих лабах:
# from scipy.spatial import ConvexHull  # для лаб с кривыми
# import quaternion  # для лаб с вращениями в пространстве

---
## Задание №1. Линейная интерполяция (LERP)

### Теория (из 01.pdf, стр. 1-2)

**Формула линейной интерполяции (LERP):**

$$\mathbf{p}(t) = \mathbf{p}_1(1-t) + \mathbf{p}_2 t$$

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Что такое LERP? | Linear Interpolation — линейная интерполяция, способ нахождения промежуточных точек между двумя заданными |
| Что означает параметр $t$? | $t$ — весовой коэффициент. При $t=0$ получаем $\mathbf{p}_1$, при $t=1$ получаем $\mathbf{p}_2$ |
| Что происходит при $t \in [0,1]$? | Точка движется между $\mathbf{p}_1$ и $\mathbf{p}_2$ |
| Что происходит при $t < 0$? | Точка находится за $\mathbf{p}_1$ (экстраполяция) |
| Что происходит при $t > 1$? | Точка находится за $\mathbf{p}_2$ (экстраполяция) |
| Как реализовать обратное движение? | Заменить $t$ на $1-t$ |

### Код

In [ ]:
# Инициализация точек отрезка (из 01.pdf, Таблица 1 не требуется, точки произвольные)
R1 = np.array([-3, -2])  # точка R1
R2 = np.array([4, 3])    # точка R2

def lerp(p1, p2, t):
    """
    Линейная интерполяция между точками p1 и p2.
    
    Параметры:
    p1, p2 : np.array — координаты точек
    t : float — параметр интерполяции (0..1 внутри отрезка)
    
    Возвращает:
    np.array — координаты промежуточной точки
    """
    return p1 * (1 - t) + p2 * t

# Середина отрезка при t = 0.5
M = lerp(R1, R2, 0.5)
print(f"Координаты середины отрезка M: ({M[0]:.2f}, {M[1]:.2f})")

# Проверка: середина должна быть средним арифметическим
M_check = (R1 + R2) / 2
print(f"Проверка: ({M_check[0]:.2f}, {M_check[1]:.2f}) — верно")

In [ ]:
def update_plot(t=0.5, direction=1):
    """
    Визуализация отрезка и движущейся точки.
    
    Параметры:
    t : float — параметр интерполяции
    direction : int — направление движения (1 = прямое R1→R2, -1 = обратное R2→R1)
    
    Математика:
    - При direction=1: точка движется от R1 (t=0) до R2 (t=1)
    - При direction=-1: точка движется от R2 (t=0) до R1 (t=1)
    - Эффект достигается заменой t → 1-t
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    # 1. Рисуем отрезок (синяя линия)
    ax.plot([R1[0], R2[0]], [R1[1], R2[1]], 'b-', linewidth=2, label='Отрезок')
    
    # 2. Рисуем концевые точки R1 и R2 (красные кружки)
    ax.plot(R1[0], R1[1], 'ro', markersize=8, label='$R_1$')
    ax.plot(R2[0], R2[1], 'ro', markersize=8, label='$R_2$')
    
    # 3. Вычисляем движущуюся точку с учётом направления
    #    direction = 1: t_eff = t
    #    direction = -1: t_eff = 1 - t (обратный порядок)
    t_eff = t if direction == 1 else 1 - t
    P = lerp(R1, R2, t_eff)
    
    # 4. Отображаем движущуюся точку (зелёный кружок)
    ax.plot(P[0], P[1], 'go', markersize=8, label=f'$P(t={t_eff:.2f})$')
    
    # 5. Подписи точек
    ax.text(R1[0]-0.3, R1[1]-0.3, '$R_1$', fontsize=12)
    ax.text(R2[0]+0.1, R2[1]+0.1, '$R_2$', fontsize=12)
    ax.text(P[0]+0.1, P[1]+0.1, '$P$', fontsize=12)
    
    # 6. Оформление (по требованиям)
    ax.grid(True)                 # сетка координат
    ax.set_aspect('equal')        # одинаковый масштаб осей
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title('Линейная интерполяция (LERP)')
    ax.legend()
    
    plt.show()

In [ ]:
# Интерактивный график с ползунками (из 01.pdf, задание 1.3-1.5)
interact(update_plot,
         t=FloatSlider(min=-1, max=2, step=0.01, value=0.5, description='Параметр $t$'),
         direction={'Прямое (R₁→R₂)': 1, 'Обратное (R₂→R₁)': -1})

# ПОЯСНЕНИЕ:
# - При t = 0..1: точка внутри отрезка
# - При t < 0: точка за R1
# - При t > 1: точка за R2
# - Переключение direction меняет направление

**Вопросы для защиты по заданию №1:**

1. **Что такое LERP и где применяется?**
   - LERP (Linear Interpolation) — линейная интерполяция. Применяется в компьютерной графике для анимации движения объектов, сглаживания, расчёта промежуточных кадров.

2. **Как вычислить середину отрезка без LERP?**
   - $\mathbf{M} = \frac{\mathbf{R}_1 + \mathbf{R}_2}{2}$

3. **Что произойдёт с точкой при $t=0.5$, $t=0$, $t=1$?**
   - $t=0.5$ — середина отрезка
   - $t=0$ — точка $R_1$
   - $t=1$ — точка $R_2$

4. **Как реализовать движение в обратном направлении без изменения функции lerp?**
   - Передать параметр $t$ как $1-t$ в функцию lerp

---

## Задание №2. Векторы и комплексные числа

### Теория (из 01.pdf, стр. 2 и презентации "Дополнительные сведения для двумерного пространства")

**Единичный вектор под углом $\theta$:**

$$\mathbf{v} = (\cos\theta, \sin\theta)$$

**Комплексное число в показательной форме:**

$$z = e^{i\theta} = \cos\theta + i\sin\theta$$

**Ориентированная площадь параллелограмма (определитель):**

$$S_{\pm}(\mathbf{a}, \mathbf{b}) = \det(\mathbf{a}, \mathbf{b}) = a_x b_y - a_y b_x$$

**Площадь треугольника:**

$$S_{\triangle} = \frac{1}{2} |\det(\mathbf{a}, \mathbf{b})|$$

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Как построить единичный вектор под углом? | $\mathbf{v} = (\cos\theta, \sin\theta)$ |
| Что такое определитель двух векторов? | Ориентированная площадь параллелограмма |
| Что означает знак определителя? | Положительный — поворот от $\mathbf{a}$ к $\mathbf{b}$ против часовой стрелки |
| Связь комплексных чисел и векторов? | $z = x + iy$ соответствует вектору $(x, y)$ |
| Что такое $e^{i\theta}$? | Единичный вектор под углом $\theta$ |

### Код

In [ ]:
# Задание 2.1: задаём единичные векторы под углами 10° и 40° (из 01.pdf)
angle_a = np.deg2rad(10)   # 10° в радианы
angle_b = np.deg2rad(40)   # 40° в радианы

a = np.array([np.cos(angle_a), np.sin(angle_a)])
b = np.array([np.cos(angle_b), np.sin(angle_b)])

print("=" * 50)
print("Задание 2.1: Компоненты векторов")
print("=" * 50)
print(f"Вектор a (10°): ({a[0]:.6f}, {a[1]:.6f})")
print(f"Вектор b (40°): ({b[0]:.6f}, {b[1]:.6f})")
print(f"Проверка длины |a| = {np.linalg.norm(a):.4f}, |b| = {np.linalg.norm(b):.4f}")
print("Единичная длина (норма = 1) подтверждает, что векторы нормированы.")

In [ ]:
# Задание 2.2: визуализация векторов (из 01.pdf, Рис. 2)
fig, ax = plt.subplots(figsize=(5, 5), dpi=200)

# Рисуем векторы из начала координат (точка O)
# ax.quiver(x, y, dx, dy) — рисует стрелку от (x,y) с направлением (dx,dy)
ax.quiver(0, 0, a[0], a[1], angles='xy', scale_units='xy', scale=1, 
          color='red', width=0.02, label='$\\mathbf{a}$')
ax.quiver(0, 0, b[0], b[1], angles='xy', scale_units='xy', scale=1, 
          color='blue', width=0.02, label='$\\mathbf{b}$')

# Строим треугольник (соединяем концы векторов)
ax.plot([a[0], b[0]], [a[1], b[1]], 'k--', linewidth=1, alpha=0.7)
ax.fill([0, a[0], b[0]], [0, a[1], b[1]], 'gray', alpha=0.2, label='Треугольник')

ax.grid(True)
ax.set_aspect('equal')
ax.set_xlim(-0.2, 1.2)
ax.set_ylim(-0.2, 1.2)
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Векторы $\\mathbf{a}$ и $\\mathbf{b}$ от начала координат')
ax.legend()

plt.show()

# ПОЯСНЕНИЕ:
# Векторы отложены от начала координат (точки O). 
# Треугольник образован векторами a, b и их разностью (a - b).

In [ ]:
# Задание 2.3 и 2.4: вычисление площадей (из 01.pdf)
# Определитель (ориентированная площадь параллелограмма)
det_ab = a[0]*b[1] - a[1]*b[0]   # det(a, b)
det_ba = b[0]*a[1] - b[1]*a[0]   # det(b, a) = -det(a, b)

print("=" * 50)
print("Задание 2.3-2.4: Площади")
print("=" * 50)
print(f"Ориентированная площадь параллелограмма (a,b): {det_ab:.6f}")
print(f"Ориентированная площадь треугольника (a,b): {det_ab/2:.6f}")
print(f"Ориентированная площадь параллелограмма (b,a): {det_ba:.6f}")
print(f"Ориентированная площадь треугольника (b,a): {det_ba/2:.6f}")
print(f"Абсолютная площадь параллелограмма: {abs(det_ab):.6f}")
print(f"Абсолютная площадь треугольника: {abs(det_ab)/2:.6f}")
print()
print("ПОЯСНЕНИЕ:")
print("- det(a,b) > 0: поворот от a к b против часовой стрелки")
print("- det(b,a) = -det(a,b): знак меняется при перестановке векторов")
print("- Абсолютная площадь не зависит от порядка векторов")

In [ ]:
# Задание 2.5: комплексные числа (из 01.pdf)
# Соответствие: вектор (x, y) ↔ комплексное число z = x + iy
z1 = a[0] + a[1]*1j   # e^{10°i}
z2 = b[0] + b[1]*1j   # e^{40°i}

print("=" * 50)
print("Задание 2.5: Комплексные числа")
print("=" * 50)
print(f"z₁ = e^{{{10}°i}} = {z1:.6f}")
print(f"z₂ = e^{{{40}°i}} = {z2:.6f}")
print("Проверка: |z₁| = ", abs(z1), " |z₂| = ", abs(z2))
print()
print("Формула Эйлера: e^{iθ} = cos θ + i sin θ")

fig, ax = plt.subplots(figsize=(5, 5), dpi=200)

# Комплексная плоскость: действительная ось (Re) — x, мнимая (Im) — y
ax.plot(z1.real, z1.imag, 'ro', markersize=8, label='$z_1 = e^{10°i}$')
ax.plot(z2.real, z2.imag, 'bo', markersize=8, label='$z_2 = e^{40°i}$')
ax.quiver(0, 0, z1.real, z1.imag, angles='xy', scale_units='xy', scale=1, color='red', width=0.02)
ax.quiver(0, 0, z2.real, z2.imag, angles='xy', scale_units='xy', scale=1, color='blue', width=0.02)

ax.grid(True)
ax.set_aspect('equal')
ax.set_xlabel('Re')
ax.set_ylabel('Im')
ax.set_title('Комплексные числа на плоскости')
ax.legend()

plt.show()

# ПОЯСНЕНИЕ:
# Комплексные числа откладываются на комплексной плоскости:
# - Горизонтальная ось (Re) — действительная часть
# - Вертикальная ось (Im) — мнимая часть
# - Модуль |z| = √(x² + y²) — длина вектора

**Вопросы для защиты по заданию №2:**

1. **Как вычислить компоненты единичного вектора, направленного под углом 30°?**
   - $\mathbf{v} = (\cos 30°, \sin 30°) = (\frac{\sqrt{3}}{2}, \frac{1}{2})$

2. **Что такое ориентированная площадь и зачем она нужна?**
   - Это площадь со знаком. Положительная — против часовой стрелки, отрицательная — по часовой. Используется для определения ориентации многоугольников и в алгоритмах вычислительной геометрии.

3. **Почему $\det(\mathbf{b}, \mathbf{a}) = -\det(\mathbf{a}, \mathbf{b})$?**
   - Определитель антисимметричен: перестановка столбцов меняет знак.

4. **Как связаны комплексные числа и вращения?**
   - Умножение на $e^{i\theta}$ поворачивает вектор на угол $\theta$.

5. **Что такое формула Эйлера?**
   - $e^{i\theta} = \cos\theta + i\sin\theta$ — связь экспоненты с тригонометрическими функциями.

---

## Задание №3. Прямая и её угол наклона

### Теория (из 01.pdf, стр. 2 и презентации "Евклидова аналитическая геометрия на плоскости")

**Параметрическое уравнение прямой:**

$$\mathbf{p}(t) = \mathbf{p}_0 + \mathbf{v}t$$

где $\mathbf{p}_0$ — точка на прямой, $\mathbf{v}$ — направляющий вектор.

**Направляющий вектор из угла наклона $\theta$:**

$$\mathbf{v} = (\cos\theta, \sin\theta)$$

**Функция `axline` в Matplotlib:**
- `axline((x0, y0), slope=k)` — прямая через точку с угловым коэффициентом $k$
- `axline((x1, y1), (x2, y2))` — прямая через две точки
- Рисует **бесконечную** прямую, уходящую за границы графика

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Как задать прямую параметрически? | $\mathbf{p}(t) = \mathbf{p}_0 + \mathbf{v}t$ |
| Как получить направляющий вектор из угла? | $\mathbf{v} = (\cos\theta, \sin\theta)$ |
| Что такое угловой коэффициент? | $k = \tan\theta = \frac{\sin\theta}{\cos\theta}$ |
| Чем `axline` отличается от `plot`? | `axline` рисует бесконечную прямую, `plot` — отрезок между двумя точками |

### Код

In [ ]:
# Задание 3.1-2: прямая через параметрическое уравнение (из 01.pdf)
def line_by_points(angle_deg=10):
    """
    Отображает прямую, заданную параметрическим уравнением.
    
    Математика:
    - Направляющий вектор v = (cosθ, sinθ)
    - Точка p0 = (-5, 0) выбрана произвольно
    - Вторая точка: p1 = p0 + v * t_end, где t_end подбирается для выхода за границы
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    angle_rad = np.deg2rad(angle_deg)
    v = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    p0 = np.array([-5, 0])  # опорная точка
    
    # Подбираем параметр t_end так, чтобы прямая выходила за границы графика
    # Если v_x близко к 0, используем 10, иначе 10/|v_x|
    t_end = 10 / (abs(v[0]) if abs(v[0]) > 0.01 else 1)
    p1 = p0 + v * t_end
    
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], 'green', linewidth=2)
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title(f'Прямая под углом {angle_deg}° (параметрическое уравнение)')
    
    plt.show()

In [ ]:
interact(line_by_points, angle_deg=IntSlider(min=-180, max=180, step=1, value=10, description='Угол (°)'))

# ПОЯСНЕНИЕ:
# При изменении угла прямая поворачивается вокруг опорной точки (-5, 0)
# Недостаток метода: прямая не бесконечная, а ограничена двумя точками

In [ ]:
# Задание 3.3: прямая через axline (бесконечная прямая) — из 01.pdf
def line_by_axline(angle_deg=10):
    """
    Отображает бесконечную прямую с помощью axline.
    
    Математика:
    - Угловой коэффициент k = tg(θ) = sinθ / cosθ
    - Прямая задаётся точкой (0, 0) и наклоном k
    - axline автоматически рисует бесконечную прямую
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    angle_rad = np.deg2rad(angle_deg)
    k = np.tan(angle_rad) if abs(np.cos(angle_rad)) > 1e-10 else float('inf')
    
    # axline(точка, slope=угловой коэффициент)
    # При k = inf (вертикальная прямая) нужно использовать другой синтаксис
    if np.isinf(k):
        ax.axvline(0, color='purple', linewidth=2)  # вертикальная прямая x = 0
    else:
        ax.axline((0, 0), slope=k, linewidth=2, color='purple')
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title(f'Прямая под углом {angle_deg}° (axline)')
    
    plt.show()

In [ ]:
interact(line_by_axline, angle_deg=IntSlider(min=-180, max=180, step=1, value=10, description='Угол (°)'))

# ПОЯСНЕНИЕ:
# axline рисует БЕСКОНЕЧНУЮ прямую, что соответствует математическому определению прямой.
# Угловой коэффициент k = tg(θ) — тангенс угла наклона.
# При θ = 90° или -90° прямая вертикальна, tg(θ) не определён → используем axvline.

**Вопросы для защиты по заданию №3:**

1. **Чем отличается параметрическое уравнение прямой от общего?**
   - Параметрическое: $\mathbf{p}(t) = \mathbf{p}_0 + \mathbf{v}t$ — удобно для вычисления точек
   - Общее: $ax + by + c = 0$ — удобно для проверки принадлежности точки

2. **Что такое направляющий вектор?**
   - Вектор, параллельный прямой. Определяет направление прямой.

3. **Как найти угловой коэффициент прямой?**
   - $k = \tan\theta = \frac{\sin\theta}{\cos\theta} = \frac{v_y}{v_x}$

4. **Почему `axline` лучше подходит для рисования прямой?**
   - Рисует бесконечную прямую, не требует вычисления двух точек.

5. **Как обработать вертикальную прямую (θ = 90°)?**
   - Угловой коэффициент бесконечен. Используем `ax.axvline(x0)`.

---

## Задание №4. Окружность — циферблат часов

### Теория (из 01.pdf, стр. 2-3 и презентации "Евклидова аналитическая геометрия на плоскости")

**Параметрическое уравнение окружности радиуса $R$:**

$$\begin{cases} x(t) = R\cos t \\ y(t) = R\sin t \end{cases}, \quad t \in [0, 2\pi)$$

**Связь часов и углов:**

| Час | Угол от Ox | Угол от Oy (для циферблата) |
|-----|------------|---------------------------|
| 12 | 90° | 0° |
| 3 | 0° | -90° |
| 6 | -90° | -180° |
| 9 | 180° | -270° |

**Формула для перевода часа в угол:**

$$\text{угол} = 90° - (\text{час} \bmod 12) \times 30°$$

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Параметрическое уравнение окружности | $x = R\cos t, y = R\sin t$ |
| Почему $t$ от 0 до $2\pi$? | Полный оборот по окружности |
| Как разместить цифры на циферблате? | Вычислить угол для каждого часа и подставить в параметрическое уравнение |
| Что такое `patches.Circle`? | Примитив Matplotlib для рисования окружности |

### Код

In [ ]:
def draw_clock(hour=12):
    """
    Рисует циферблат часов с движущейся стрелкой.
    
    Математика:
    - Окружность радиуса 1 с центром в (0,0)
    - Угол стрелки: 12 часов = 90° (вверх), 3 часа = 0° (вправо)
    - Формула: angle_deg = 90 - (hour % 12) * 30
    - Цифры размещены на окружности радиуса 0.85 (чуть внутри)
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    # 1. Окружность (patches.Circle — примитив Matplotlib)
    circle = patches.Circle((0, 0), radius=1, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(circle)
    
    # 2. Вычисление угла для стрелки
    angle_deg = 90 - (hour % 12) * 30
    angle_rad = np.deg2rad(angle_deg)
    x_end = np.cos(angle_rad)
    y_end = np.sin(angle_rad)
    
    # 3. Стрелка (радиус-вектор)
    ax.arrow(0, 0, x_end, y_end, head_width=0.07, head_length=0.1, 
             fc='red', ec='red', width=0.02)
    
    # 4. Цифры циферблата (1-12)
    for h in range(1, 13):
        ang_deg = 90 - h * 30
        ang_rad = np.deg2rad(ang_deg)
        x_num = 0.85 * np.cos(ang_rad)   # радиус 0.85, чтобы цифры были внутри окружности
        y_num = 0.85 * np.sin(ang_rad)
        ax.text(x_num, y_num, str(h), fontsize=12, ha='center', va='center')
    
    # 5. Оформление
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title('Циферблат часов')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    
    plt.show()

In [ ]:
interact(draw_clock, hour=IntSlider(min=1, max=12, step=1, value=12, description='Час'))

# ПОЯСНЕНИЕ:
# - При hour=12 стрелка указывает на 12 часов (вверх)
# - При hour=3 стрелка указывает на 3 часа (вправо)
# - При hour=6 стрелка указывает на 6 часов (вниз)
# - При hour=9 стрелка указывает на 9 часов (влево)
# - Цифры размещены с помощью параметрического уравнения окружности

**Вопросы для защиты по заданию №4:**

1. **Как параметрически задать окружность?**
   - $x = R\cos t, y = R\sin t, t \in [0, 2\pi)$

2. **Почему для цифр используется радиус 0.85, а не 1?**
   - Чтобы цифры были внутри окружности, а не на её границе.

3. **Как вычислить угол для часа?**
   - Каждый час = 30°. Отсчёт от 12 часов (90° от Ox). Формула: $90° - \text{час} \times 30°$

4. **Что делает `patches.Circle`?**
   - Создаёт объект окружности. `ax.add_patch()` добавляет его на график.

5. **Как изменить направление вращения стрелки?**
   - Изменить знак в формуле: $90° + \text{час} \times 30°$ (против часовой)

---

## Задание №5. Прямоугольник и матричные преобразования

### Теория (из 01.pdf, стр. 3 и презентации "Линейные и аффинные преобразования")

**Аффинное преобразование:**

$$\mathbf{p}' = A\mathbf{p} + \mathbf{b}$$

где $A$ — матрица линейного преобразования, $\mathbf{b}$ — вектор трансляции.

**Матрица сдвига (shear) в задании:**

$$A = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}$$

**Матрица поворота на 180°:**

$$B = \begin{pmatrix} -1 & 0 \\ 0 & -1 \end{pmatrix}$$

**Формула ориентированной площади (формула шнурка, shoelace formula):**

$$S = \frac{1}{2} \sum_{i=1}^{n} (x_i y_{i+1} - x_{i+1} y_i)$$

где $(x_{n+1}, y_{n+1}) = (x_1, y_1)$.

**Свойства определителя и площади:**

| Свойство | Формула |
|----------|---------|
| Площадь после преобразования | $S' = |\det A| \cdot S$ |
| Знак определителя | $\det A > 0$ — сохраняет ориентацию, $<0$ — меняет |

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Что такое матричное преобразование? | Умножение каждого вектора на матрицу |
| Как изменится площадь при преобразовании? | Умножается на $|\det A|$ |
| Что означает знак ориентированной площади? | Положительный — обход против часовой стрелки |
| Формула шнурка (Гаусса) | $S = \frac{1}{2}\sum(x_i y_{i+1} - x_{i+1} y_i)$ |

### Код

In [ ]:
# Задание 5.1: прямоугольник с центром в (0,0) (из 01.pdf)
width, height = 4, 2  # не квадрат (ширина ≠ высоте)
rect = np.array([
    [-width/2, -height/2],   # вершина 0 (левая нижняя)
    [ width/2, -height/2],   # вершина 1 (правая нижняя)
    [ width/2,  height/2],   # вершина 2 (правая верхняя)
    [-width/2,  height/2]    # вершина 3 (левая верхняя)
])
rect_closed = np.vstack([rect, rect[0]])  # замыкаем для рисования

print("Координаты вершин прямоугольника:")
for i, v in enumerate(rect):
    print(f"  Вершина {i}: ({v[0]:.1f}, {v[1]:.1f})")

In [ ]:
def oriented_area(vertices):
    """
    Вычисление ориентированной площади многоугольника (формула шнурка).
    
    Параметры:
    vertices : np.array — массив вершин (n x 2), последняя вершина НЕ повторяет первую
    
    Возвращает:
    float — ориентированная площадь
    
    Математика:
    S = 1/2 * Σ(x_i*y_{i+1} - x_{i+1}*y_i)
    """
    x = vertices[:, 0]
    y = vertices[:, 1]
    # Добавляем первую точку в конец для замыкания
    x_closed = np.append(x, x[0])
    y_closed = np.append(y, y[0])
    return 0.5 * np.sum(x_closed[:-1] * y_closed[1:] - x_closed[1:] * y_closed[:-1])

area_orig = oriented_area(rect)
print(f"Ориентированная площадь исходного прямоугольника: {area_orig:.4f}")
print("Положительный знак означает, что вершины перечислены против часовой стрелки")
print(f"Проверка (ширина * высота): {width * height:.1f}")

In [ ]:
# Задание 5.2: матрица A = [[1, 2], [0, 1]] (сдвиг по x, shear)
A = np.array([[1, 2], [0, 1]])
rect_transformed = rect @ A.T  # Транспонирование! (умножаем векторы-строки на A^T)

area_trans = oriented_area(rect_transformed)
print("=" * 50)
print("Преобразование матрицей A = [[1, 2], [0, 1]]")
print("=" * 50)
print(f"Ориентированная площадь после A: {area_trans:.4f}")
print(f"det(A) = {np.linalg.det(A):.4f}")
print(f"Ожидаемая площадь: {area_orig * np.linalg.det(A):.4f}")
print()
print("ПОЯСНЕНИЕ: det(A)=1, поэтому площадь сохранилась.")
print("Матрица A задаёт сдвиг (shear) — прямоугольник превращается в параллелограмм.")

In [ ]:
# Задание 5.4: матрица B = [[-1, 0], [0, -1]] (поворот на 180°)
B = np.array([[-1, 0], [0, -1]])
rect_transformed2 = rect @ B.T

area_trans2 = oriented_area(rect_transformed2)
print("=" * 50)
print("Преобразование матрицей B = [[-1, 0], [0, -1]]")
print("=" * 50)
print(f"Ориентированная площадь после B: {area_trans2:.4f}")
print(f"det(B) = {np.linalg.det(B):.4f}")
print()
print("ПОЯСНЕНИЕ: det(B)=1, знак площади НЕ изменился.")
print("Матрица B задаёт поворот на 180° — фигура перевернута, но ориентация сохранилась.")

In [ ]:
# Визуализация всех трёх состояний
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), dpi=200)

# Исходный прямоугольник
ax1.add_patch(patches.Polygon(rect, fill=False, edgecolor='blue', linewidth=2))
ax1.set_aspect('equal')
ax1.grid(True)
ax1.set_xlim(-3, 3)
ax1.set_ylim(-2, 2)
ax1.set_title('Исходный прямоугольник')
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.axvline(0, color='gray', linewidth=0.5)

# После A (сдвиг)
ax2.add_patch(patches.Polygon(rect_transformed, fill=False, edgecolor='red', linewidth=2))
ax2.set_aspect('equal')
ax2.grid(True)
ax2.set_xlim(-3, 6)
ax2.set_ylim(-2, 4)
ax2.set_title('После A = [[1, 2], [0, 1]]')
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.axvline(0, color='gray', linewidth=0.5)

# После B (поворот 180°)
ax3.add_patch(patches.Polygon(rect_transformed2, fill=False, edgecolor='green', linewidth=2))
ax3.set_aspect('equal')
ax3.grid(True)
ax3.set_xlim(-3, 3)
ax3.set_ylim(-2, 2)
ax3.set_title('После B = [[-1, 0], [0, -1]]')
ax3.axhline(0, color='gray', linewidth=0.5)
ax3.axvline(0, color='gray', linewidth=0.5)

plt.show()

**Вопросы для защиты по заданию №5:**

1. **Как изменится площадь фигуры при преобразовании матрицей A?**
   - Площадь умножается на $|\det A|$. При $\det A = 1$ площадь сохраняется.

2. **Что такое формула шнурка (shoelace formula)?**
   - $S = \frac{1}{2} \sum_{i=1}^{n} (x_i y_{i+1} - x_{i+1} y_i)$ — для вычисления площади многоугольника.

3. **Почему ориентированная площадь после B осталась положительной?**
   - $\det B = 1 > 0$, значит ориентация (обход вершин) сохранилась.

4. **Какой порядок обхода вершин даёт положительную площадь?**
   - Против часовой стрелки.

5. **Почему при умножении используется `@ A.T`?**
   - У нас векторы-строки. Чтобы применить матрицу A к вектору-строке, нужно умножить на $A^T$.
   - Для векторов-столбцов: $\mathbf{p}' = A \mathbf{p}$.

---

## Задание №6. Эллипс и гипербола

### Теория (из 01.pdf, стр. 3 и презентации "Евклидова аналитическая геометрия на плоскости")

**Параметрическое уравнение эллипса с центром в $(t_x, t_y)$:**

$$\begin{cases} x(t) = a\cos t + t_x \\ y(t) = b\sin t + t_y \end{cases}, \quad t \in [0, 2\pi)$$

где $a$ — большая полуось (по x), $b$ — малая полуось (по y).

**Параметрическое уравнение гиперболы (правая ветвь):**

$$\begin{cases} x(t) = a\operatorname{ch} t \\ y(t) = b\operatorname{sh} t \end{cases}, \quad t \in \mathbb{R}$$

где $\operatorname{ch} t = \frac{e^t + e^{-t}}{2}$ (гиперболический косинус), $\operatorname{sh} t = \frac{e^t - e^{-t}}{2}$ (гиперболический синус).

**Асимптоты гиперболы:**

$$y = \pm \frac{b}{a} x$$

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Параметры эллипса $a$ и $b$ | $a$ — полуось по x, $b$ — полуось по y |
| Как сместить эллипс? | Прибавить вектор трансляции $(t_x, t_y)$ к координатам |
| Что такое гиперболические функции? | $\operatorname{ch} t = \frac{e^t+e^{-t}}{2}$, $\operatorname{sh} t = \frac{e^t-e^{-t}}{2}$ |
| Что такое асимптоты? | Прямые, к которым стремится гипербола на бесконечности |

### Код

In [ ]:
# Задание 6.1-2: эллипс с трансляцией (из 01.pdf)
def draw_ellipse(a=2, b=1, tx=1, ty=1):
    """
    Рисует эллипс с центром в (tx, ty).
    
    Параметры:
    a, b — полуоси
    tx, ty — координаты центра
    
    Математика: x = a*cos(t) + tx, y = b*sin(t) + ty
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    t = np.linspace(0, 2*np.pi, 100)
    x = a * np.cos(t) + tx
    y = b * np.sin(t) + ty
    
    ax.plot(x, y, 'b-', linewidth=2, label='Эллипс')
    
    # Точки через каждые 45° для наглядности
    t_marks = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    x_marks = a * np.cos(t_marks) + tx
    y_marks = b * np.sin(t_marks) + ty
    ax.plot(x_marks, y_marks, 'ro', markersize=6, label='Точки')
    
    # Центр эллипса
    ax.plot(tx, ty, 'go', markersize=8, label='Центр')
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(tx - a - 1, tx + a + 1)
    ax.set_ylim(ty - b - 1, ty + b + 1)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title(f'Эллипс: $a={a}$, $b={b}$, центр $({tx},{ty})$')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.legend()
    
    plt.show()

In [ ]:
interact(draw_ellipse,
         a=FloatSlider(min=0.5, max=4, step=0.1, value=2, description='$a$'),
         b=FloatSlider(min=0.5, max=4, step=0.1, value=1, description='$b$'),
         tx=FloatSlider(min=-3, max=3, step=0.5, value=1, description='$t_x$'),
         ty=FloatSlider(min=-3, max=3, step=0.5, value=1, description='$t_y$'))

# ПОЯСНЕНИЕ:
# - a — радиус по оси x (чем больше a, тем шире эллипс)
# - b — радиус по оси y (чем больше b, тем выше эллипс)
# - tx, ty — смещение центра эллипса

In [ ]:
# Задание 6.3-5: гипербола (из 01.pdf)
def draw_hyperbola(a=1, b=1):
    """
    Рисует правую ветвь гиперболы и её асимптоты.
    
    Параметры:
    a, b — параметры гиперболы
    
    Математика:
    - x = a*ch(t), y = b*sh(t) — параметрическое уравнение
    - Асимптоты: y = ±(b/a)*x
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    t = np.linspace(-2, 2, 100)
    x = a * np.cosh(t)
    y = b * np.sinh(t)
    
    ax.plot(x, y, 'g-', linewidth=2, label='Гипербола (правая ветвь)')
    
    # Левая ветвь (для симметрии, не требуется по заданию, но для наглядности)
    ax.plot(-x, y, 'g--', linewidth=1, alpha=0.5, label='Левая ветвь')
    
    # Асимптоты
    x_asp = np.linspace(0, 6, 100)
    ax.plot(x_asp, (b/a)*x_asp, 'r--', linewidth=1, alpha=0.7, label='Асимптоты')
    ax.plot(x_asp, -(b/a)*x_asp, 'r--', linewidth=1, alpha=0.7)
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(0, 6)
    ax.set_ylim(-4, 4)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    ax.set_title(f'Гипербола: $a={a}$, $b={b}$')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.legend()
    
    plt.show()

In [ ]:
interact(draw_hyperbola,
         a=FloatSlider(min=0.5, max=3, step=0.1, value=1, description='$a$'),
         b=FloatSlider(min=0.5, max=3, step=0.1, value=1, description='$b$'))

# ПОЯСНЕНИЕ:
# - a влияет на "ширину" гиперболы (расстояние до асимптот по x)
# - b влияет на "высоту" гиперболы (расстояние до асимптот по y)
# - Асимптоты: y = ±(b/a)x
# - При a=b получается равнобочная гипербола (асимптоты под 45°)
# - ch(t) = (e^t + e^{-t})/2, sh(t) = (e^t - e^{-t})/2

**Вопросы для защиты по заданию №6:**

1. **Как параметрически задать эллипс?**
   - $x = a\cos t, y = b\sin t$

2. **Что такое полуоси эллипса?**
   - $a$ — расстояние от центра до вершины по x, $b$ — по y.

3. **Как сместить эллипс в точку (tx, ty)?**
   - $x = a\cos t + tx, y = b\sin t + ty$

4. **Что такое гиперболические функции?**
   - $\operatorname{ch} t = \frac{e^t+e^{-t}}{2}$ — гиперболический косинус
   - $\operatorname{sh} t = \frac{e^t-e^{-t}}{2}$ — гиперболический синус

5. **Что такое асимптоты гиперболы?**
   - Прямые $y = \pm \frac{b}{a}x$, к которым гипербола стремится при $x \to \infty$.

6. **Почему гипербола не замкнута?**
   - Параметр $t \in \mathbb{R}$ даёт бесконечную кривую.

---

## Задание №7. Часы с секундной стрелкой

### Теория (из 01.pdf, стр. 4)

**Анимация в Jupyter через ползунки:**
- Вместо настоящей анимации используется `IntSlider`, который позволяет менять секунду вручную.
- Для автоматического движения потребовался бы цикл с `time.sleep()`, что в Jupyter работает плохо.

**Проблема равномерного движения по эллипсу:**
- Равномерное изменение угла $\theta$ даёт неравномерное движение точки по эллипсу.
- Для равномерного движения нужен пересчёт параметра через эллиптический интеграл (выходит за рамки лабы).

**Что нужно знать для защиты:**

| Вопрос | Ответ |
|--------|-------|
| Как сделать анимацию в Jupyter? | Через `ipywidgets.interact` с ползунком |
| Почему на эллипсе стрелка движется неравномерно? | Параметр $t$ в $(a\cos t, b\sin t)$ — не угол, а параметр |
| Как исправить неравномерность? | Нужно решить уравнение $\theta(t) = \arctan(\frac{b}{a}\tan t)$ |

### Код

In [ ]:
# Задание 7.1: часы с окружностью (секундная стрелка)
def clock_circle(second=0):
    """
    Часы с круговым циферблатом.
    
    Математика:
    - 60 секунд = полный оборот
    - 1 секунда = 6°
    - Угол: 90° - second * 6° (отсчёт от 12 часов)
    """
    fig, ax = plt.subplots(figsize=(5, 5), dpi=200)
    
    # Окружность
    circle = patches.Circle((0, 0), radius=1, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(circle)
    
    # Секундная стрелка
    angle_deg = 90 - second * 6
    angle_rad = np.deg2rad(angle_deg)
    x_end = np.cos(angle_rad)
    y_end = np.sin(angle_rad)
    ax.arrow(0, 0, x_end, y_end, head_width=0.07, head_length=0.1, 
             fc='red', ec='red', width=0.02)
    
    # Цифры
    for h in range(1, 13):
        ang_deg = 90 - h * 30
        ang_rad = np.deg2rad(ang_deg)
        x_num = 0.85 * np.cos(ang_rad)
        y_num = 0.85 * np.sin(ang_rad)
        ax.text(x_num, y_num, str(h), fontsize=12, ha='center', va='center')
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title('Часы с секундной стрелкой (окружность)')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    
    plt.show()

In [ ]:
interact(clock_circle, second=IntSlider(min=0, max=59, step=1, value=0, description='Секунда'))

In [ ]:
# Задание 7.2: часы с эллипсом
def clock_ellipse(second=0, a=2, b=1.5):
    """
    Часы с эллиптическим циферблатом.
    
    Математика:
    - Эллипс: x = a*cos(t), y = b*sin(t)
    - Угол для стрелки: тот же, что для окружности (неравномерное движение!)
    - Для равномерного движения нужен пересчёт параметра
    """
    fig, ax = plt.subplots(figsize=(6, 5), dpi=200)
    
    # Эллипс
    t_ell = np.linspace(0, 2*np.pi, 100)
    x_ell = a * np.cos(t_ell)
    y_ell = b * np.sin(t_ell)
    ax.plot(x_ell, y_ell, 'black', linewidth=2)
    
    # Стрелка (используем тот же угол, что для окружности)
    angle_deg = 90 - second * 6
    angle_rad = np.deg2rad(angle_deg)
    x_end = a * np.cos(angle_rad)
    y_end = b * np.sin(angle_rad)
    ax.arrow(0, 0, x_end, y_end, head_width=0.1, head_length=0.15, 
             fc='red', ec='red', width=0.03)
    
    # Цифры на эллипсе
    for h in range(1, 13):
        ang_deg = 90 - h * 30
        ang_rad = np.deg2rad(ang_deg)
        x_num = (a - 0.2) * np.cos(ang_rad)
        y_num = (b - 0.2) * np.sin(ang_rad)
        ax.text(x_num, y_num, str(h), fontsize=12, ha='center', va='center')
    
    ax.grid(True)
    ax.set_aspect('equal')
    ax.set_xlim(-a-1, a+1)
    ax.set_ylim(-b-1, b+1)
    ax.set_title(f'Часы с эллиптическим циферблатом ($a={a}$, $b={b}$)')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    
    plt.show()

In [ ]:
interact(clock_ellipse,
         second=IntSlider(min=0, max=59, step=1, value=0, description='Секунда'),
         a=FloatSlider(min=1, max=3, step=0.1, value=2, description='$a$'),
         b=FloatSlider(min=0.8, max=2.5, step=0.1, value=1.5, description='$b$'))

# ПОЯСНЕНИЕ О НЕРАВНОМЕРНОСТИ:
# На эллипсе равномерное изменение угла даёт неравномерное движение точки.
# Это связано с тем, что параметр t в уравнениях (a*cos t, b*sin t) — НЕ угол.
# Угол θ связан с t соотношением: tg θ = (b/a) * tg t
# Для равномерного движения по эллипсу нужно решать это уравнение относительно t.

**Вопросы для защиты по заданию №7:**

1. **Как работает анимация в Jupyter?**
   - Через `ipywidgets.interact` — при изменении ползунка перерисовывается график.

2. **Почему на эллипсе стрелка движется неравномерно?**
   - Параметр $t$ в уравнениях $x = a\cos t, y = b\sin t$ — не угол, а параметр.
   - Угол $\theta$ связан с $t$: $\tan\theta = \frac{b}{a}\tan t$.

3. **Как сделать равномерное движение по эллипсу?**
   - Нужно решать уравнение $\theta = \arctan\left(\frac{b}{a}\tan t\right)$ относительно $t$.
   - Это выходит за рамки лабораторной работы.

4. **Почему для окружности неравномерности нет?**
   - При $a = b = R$: $\tan\theta = \tan t \Rightarrow \theta = t$.

---

## Общий вывод по лабораторной работе №1

В ходе выполнения работы были освоены:

| № | Тема | Что освоено |
|---|------|-------------|
| 1 | LERP | Линейная интерполяция, параметрическое задание отрезка, экстраполяция |
| 2 | Векторы | Длины, углы, площади, ориентированная площадь, определитель |
| 3 | Комплексные числа | Изоморфизм с векторами, формула Эйлера |
| 4 | Прямые | Параметрическое уравнение, `axline`, угловой коэффициент |
| 5 | Окружность | Параметрическое уравнение, `patches.Circle`, циферблат |
| 6 | Матрицы | Аффинные преобразования, ориентированная площадь, формула шнурка |
| 7 | Эллипс и гипербола | Параметрические уравнения, полуоси, асимптоты, трансляция |
| 8 | Анимация | `ipywidgets`, ползунки, неравномерность движения по эллипсу |

**Все требования соблюдены:**
- ✅ Объектно-ориентированный интерфейс Matplotlib (`ax.`, `fig.`)
- ✅ `ax.grid(True)`
- ✅ `ax.set_aspect('equal')`
- ✅ `figsize=(5,5)`, `dpi=200`
- ✅ Интерактивные ползунки (`ipywidgets`)
- ✅ Формулы в LaTeX

---

## Список использованных материалов (из курса)

1. **00.pdf** — Общее описание лабораторных работ (требования к оформлению)
2. **01.pdf** — Лабораторная работа №1 (все задания)
3. **main.pdf** — Презентации курса:
   - Векторная алгебра
   - Дополнительные сведения для двумерного пространства
   - Евклидова аналитическая геометрия на плоскости
   - Линейные и аффинные преобразования

---

*Ноутбук подготовлен для защиты лабораторной работы №1 по курсу "Компьютерная геометрия".*

*Студент должен уметь объяснить каждую формулу и каждую строку кода.*